## 1. Data Preprocessing
In this section, we import the raw ESS data and prepare it for analysis by handling missing values and transforming variables.

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset
zip_path = "ESS9e03_3-ESS10e03_3-ESS10SCe03_2-ESS11e04_1-subset.zip"
csv_inside = "ESS9e03_3-ESS10e03_3-ESS10SCe03_2-ESS11e04_1-subset.csv"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open(csv_inside) as f:
        df = pd.read_csv(f, low_memory=False)

print("CSV Load Successful!")
print(f"Dataset Size: {df.shape[0]} rows x {df.shape[1]} columns")

### Data Exploration
Before cleaning, we examine the raw structure of the data, including country distribution and the default scaling of the human value variables.

In [ ]:
# Quick preview of key variables
interest_cols = ['cntry', 'yrbrn'] + [c for c in df.columns if c.startswith('ip')][:5]
display(df[interest_cols].head())

print("\n### Country Distribution (Top 10)")
display(df['cntry'].value_counts().head(10))

### Data Cleansing & Variable Transformation
To ensure the data is research-ready, we perform the following:
1. **Missing Values:** Convert ESS codes (7, 8, 9, 77, 88, 99) to `NaN`.
2. **Scale Reversal:** We transform the 1–6 scale to ensure **6 = Strong Agreement**.
3. **Generational Mapping:** Creating cohorts (Gen Z to Boomers) based on `yrbrn`.
4. **Schwartz Dimensions:** Mapping 21 items into the 10 core values.

In [ ]:
# 1. Handle Missing Data
value_cols = [c for c in df.columns if c.startswith('ip')]
df[value_cols] = df[value_cols].replace([7, 8, 9], np.nan)
df['lrscale'] = df['lrscale'].replace([77, 88, 99], np.nan)
df['yrbrn'] = df['yrbrn'].replace([9999], np.nan)

# 2. Reverse Coding (7 - x)
df[value_cols] = 7 - df[value_cols]

# 3. Create Generation Column
conditions = [
    (df['yrbrn'] >= 1997),
    (df['yrbrn'] >= 1981) & (df['yrbrn'] <= 1996),
    (df['yrbrn'] >= 1965) & (df['yrbrn'] <= 1980),
    (df['yrbrn'] <= 1964)
]
labels = ['Gen Z', 'Millennials', 'Gen X', 'Boomers']
df['generation'] = np.select(conditions, labels, default='Unknown')

# 4. Create Schwartz Value Dimensions (The Full 10)
df['Universalism'] = df[['ipudrst', 'iphlppl', 'ipenvir']].mean(axis=1)
df['Benevolence'] = df[['iphlppl', 'ipudrst']].mean(axis=1)
df['Tradition'] = df[['imptrad', 'ipmodst']].mean(axis=1)
df['Security'] = df[['impsafe', 'ipstrgv']].mean(axis=1)
df['Conformity'] = df[['ipfrule', 'ipbhvar']].mean(axis=1)
df['Self-Direction'] = df[['impfree', 'ipgdtim']].mean(axis=1)
df['Stimulation'] = df[['ipstmlt', 'iphlth']].mean(axis=1)
df['Hedonism'] = df[['ipgdtim', 'impdiff']].mean(axis=1)
df['Achievement'] = df[['ipeqopt', 'ipshlpp']].mean(axis=1)
df['Power'] = df[['iprich', 'iprspot']].mean(axis=1)

# Define the list for easy access in future cells
schwartz_values = [
    'Universalism', 'Benevolence', 'Tradition', 'Security', 'Conformity',
    'Self-Direction', 'Stimulation', 'Hedonism', 'Achievement', 'Power'
]

print("Data Cleansing Complete: Generations assigned and Schwartz 10 dimensions calculated.")

# 5. Cleaned Data Preview for the team
print("\n### 6. Cleaned Data Preview")
display(df[['generation', 'lrscale'] + schwartz_values].head())

## 2. Descriptive Analysis
Now that the data is clean, we will begin answering our primary research questions by looking at the averages across generations and regions.

In [ ]:
# Final check: Average Universalism by Generation
print("### Mean Universalism Score by Generation")
display(df.groupby('generation')['Universalism'].mean().sort_values(ascending=False))